# CS224N — 0-Shot Evaluation: Comprehensive Metrics Analysis

This notebook loads all 0-shot benchmark results and produces:
1. A unified metrics table across all models × tasks
2. Accuracy comparison bar charts
3. Throughput (tokens/sec) and efficiency analysis
4. RAFT per-task breakdown
5. Radar chart (model capability profile)
6. Accuracy vs. efficiency scatter (Pareto frontier)

In [1]:
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Try to use a nice style; fall back gracefully
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '0_shot'))
print(f'Loading metrics from: {BASE_DIR}')

Loading metrics from: /Users/danielgrossman/224nProject/CS224N-TheEfficiencyThreshold/src/0_shot


## 1. Load All Metrics

In [ ]:
# ── Model display names ────────────────────────────────────────────────────
MODEL_LABELS = {
    'meta-llama/Meta-Llama-3-8B-Instruct': 'Llama-3 8B',
    'Qwen/Qwen3-4B':                        'Qwen3 4B',
    'Qwen/Qwen3-4B-Instruct-2507':          'Qwen3 4B',
    'Qwen/Qwen3-8B':                        'Qwen3 8B',
}

MODEL_SIZES = {
    'Llama-3 8B': 8,
    'Qwen3 4B':   4,
    'Qwen3 8B':   8,
}

MODEL_COLORS = {
    'Llama-3 8B': '#4C72B0',
    'Qwen3 4B':   '#DD8452',
    'Qwen3 8B':   '#55A868',
}

# ── Task display names ─────────────────────────────────────────────────────
TASK_LABELS = {
    'financial_phrase_bank': 'Financial PhraseBank',
    'gsm8k':                 'GSM8K (Math)',
    'superglue_boolq':       'SuperGLUE BoolQ',
    'superglue_rte':         'SuperGLUE RTE',
    'raft':                  'RAFT (mean)',
}

# ── Hardcoded metrics (from all JSON files) ────────────────────────────────
# Format: (task_key, model_raw, accuracy, gen_tokens_per_sec, total_gen_time_min, eval_size)
RAW_METRICS = [
    # Financial Phrase Bank
    ('financial_phrase_bank', 'meta-llama/Meta-Llama-3-8B-Instruct', 0.906,  12.34,  2.70,  1000),
    ('financial_phrase_bank', 'Qwen/Qwen3-4B-Instruct-2507',         0.900,  24.95,  1.34,  1000),
    ('financial_phrase_bank', 'Qwen/Qwen3-8B',                        0.963, 126.76,  0.263, 1000),
    # GSM8K
    ('gsm8k',                 'meta-llama/Meta-Llama-3-8B-Instruct', 0.7445, 28.78,  95.32, 1319),
    ('gsm8k',                 'Qwen/Qwen3-8B',                        0.8764, 843.39,  6.80, 1319),
    # SuperGLUE BoolQ
    ('superglue_boolq',       'meta-llama/Meta-Llama-3-8B-Instruct', 0.8456, 44.24,   2.46, 3270),
    ('superglue_boolq',       'Qwen/Qwen3-4B',                        0.8544, 105.20,  1.51, 3270),
    ('superglue_boolq',       'Qwen/Qwen3-8B',                        0.8679, 44.24,   2.52, 3270),
    # SuperGLUE RTE
    ('superglue_rte',         'meta-llama/Meta-Llama-3-8B-Instruct', 0.7834, 172.07,  0.115, 277),
    ('superglue_rte',         'Qwen/Qwen3-4B-Instruct-2507',         0.8664, 309.84,  0.0665, 277),
    ('superglue_rte',         'Qwen/Qwen3-8B',                        0.8592, 156.46,  0.130,  277),
    # RAFT (mean across 10 tasks — Qwen3-8B only)
    ('raft',                  'Qwen/Qwen3-8B',                        0.408,  None,    None,   None),
]

rows = []
for task, model_raw, acc, tps, tgm, esz in RAW_METRICS:
    rows.append({
        'task':              task,
        'task_label':        TASK_LABELS[task],
        'model_raw':         model_raw,
        'model':             MODEL_LABELS[model_raw],
        'accuracy':          acc,
        'gen_tokens_per_sec': tps,
        'total_gen_time_min': tgm,
        'eval_size':         esz,
    })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} result entries')
df

## 2. Summary Table

In [ ]:
# Pivot: tasks as rows, models as columns
pivot_acc = df.pivot_table(
    index='task_label', columns='model', values='accuracy'
).reindex(columns=['Llama-3 8B', 'Qwen3 4B', 'Qwen3 8B'])

# Add mean accuracy across tasks for each model
pivot_acc.loc['** MEAN **'] = pivot_acc.mean()

display_acc = (pivot_acc * 100).round(2).astype(str) + '%'
display_acc = display_acc.replace('nan%', '—')
print('=== Accuracy by Model × Task ===')
print(display_acc.to_string())
display_acc

In [ ]:
# Throughput pivot
pivot_tps = df.pivot_table(
    index='task_label', columns='model', values='gen_tokens_per_sec'
).reindex(columns=['Llama-3 8B', 'Qwen3 4B', 'Qwen3 8B'])

display_tps = pivot_tps.round(1).astype(str) + ' tok/s'
display_tps = display_tps.replace('nan tok/s', '—')
print('\n=== Throughput (tokens/sec) by Model × Task ===')
print(display_tps.to_string())

## 3. Accuracy Comparison — Bar Charts

In [ ]:
# Tasks to include in grouped bar chart (exclude RAFT since only 1 model)
MAIN_TASKS = ['financial_phrase_bank', 'gsm8k', 'superglue_boolq', 'superglue_rte']
df_main = df[df['task'].isin(MAIN_TASKS)].copy()

tasks_ordered = ['Financial PhraseBank', 'GSM8K (Math)', 'SuperGLUE BoolQ', 'SuperGLUE RTE']
models_ordered = ['Llama-3 8B', 'Qwen3 4B', 'Qwen3 8B']

x = np.arange(len(tasks_ordered))
width = 0.25
offsets = [-width, 0, width]

fig, ax = plt.subplots(figsize=(13, 6))

for i, model in enumerate(models_ordered):
    vals = []
    for tl in tasks_ordered:
        row = df_main[(df_main['task_label'] == tl) & (df_main['model'] == model)]
        vals.append(row['accuracy'].values[0] * 100 if len(row) else np.nan)
    bars = ax.bar(
        x + offsets[i], vals, width,
        label=model,
        color=MODEL_COLORS[model],
        edgecolor='white', linewidth=0.8,
        alpha=0.92
    )
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold'
            )

ax.set_xticks(x)
ax.set_xticklabels(tasks_ordered, fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('0-Shot Accuracy by Model and Benchmark', fontsize=14, fontweight='bold', pad=14)
ax.set_ylim(60, 102)
ax.legend(title='Model', fontsize=10, title_fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.axhline(y=100, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_comparison.png')

## 4. Throughput (Tokens/sec) Comparison

In [ ]:
df_tps = df_main.dropna(subset=['gen_tokens_per_sec']).copy()

fig, ax = plt.subplots(figsize=(13, 6))

for i, model in enumerate(models_ordered):
    vals = []
    for tl in tasks_ordered:
        row = df_tps[(df_tps['task_label'] == tl) & (df_tps['model'] == model)]
        vals.append(row['gen_tokens_per_sec'].values[0] if len(row) else np.nan)
    bars = ax.bar(
        x + offsets[i], vals, width,
        label=model,
        color=MODEL_COLORS[model],
        edgecolor='white', linewidth=0.8,
        alpha=0.92
    )
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            label_val = f'{v:.0f}' if v >= 10 else f'{v:.1f}'
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                label_val, ha='center', va='bottom', fontsize=8, fontweight='bold'
            )

ax.set_xticks(x)
ax.set_xticklabels(tasks_ordered, fontsize=11)
ax.set_ylabel('Generated Tokens / Second', fontsize=12)
ax.set_title('0-Shot Inference Throughput by Model and Benchmark', fontsize=14, fontweight='bold', pad=14)
ax.legend(title='Model', fontsize=10, title_fontsize=10)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:,.0f}'))

plt.tight_layout()
plt.savefig('throughput_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: throughput_comparison.png')

## 5. Efficiency: Accuracy vs. Inference Time

In [ ]:
df_eff = df_main.dropna(subset=['total_gen_time_min']).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: scatter accuracy vs. gen time ───────────────────────────────────
ax = axes[0]
for model in models_ordered:
    sub = df_eff[df_eff['model'] == model]
    sc = ax.scatter(
        sub['total_gen_time_min'], sub['accuracy'] * 100,
        label=model, color=MODEL_COLORS[model],
        s=120, zorder=5, edgecolors='white', linewidths=1.2
    )
    for _, row in sub.iterrows():
        ax.annotate(
            row['task_label'],
            (row['total_gen_time_min'], row['accuracy'] * 100),
            xytext=(6, 3), textcoords='offset points',
            fontsize=7.5, color=MODEL_COLORS[model]
        )

ax.set_xlabel('Total Generation Time (minutes)', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Accuracy vs. Inference Time\n(lower-left = faster, upper-left = best)', fontsize=11, fontweight='bold')
ax.legend(title='Model', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

# ── Right: accuracy-per-minute (efficiency score) ─────────────────────────
df_eff['acc_per_min'] = df_eff['accuracy'] / df_eff['total_gen_time_min']

ax2 = axes[1]
eff_plot = df_eff.sort_values('acc_per_min', ascending=True)
colors_bar = [MODEL_COLORS[m] for m in eff_plot['model']]
labels_bar = eff_plot['task_label'] + '\n(' + eff_plot['model'] + ')'

bars = ax2.barh(labels_bar, eff_plot['acc_per_min'], color=colors_bar, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, eff_plot['acc_per_min']):
    ax2.text(v + 0.01, bar.get_y() + bar.get_height() / 2,
             f'{v:.2f}', va='center', ha='left', fontsize=8.5, fontweight='bold')

ax2.set_xlabel('Accuracy / Minute (higher = more efficient)', fontsize=11)
ax2.set_title('Inference Efficiency Score\n(accuracy ÷ generation time)', fontsize=11, fontweight='bold')
legend_patches = [mpatches.Patch(color=MODEL_COLORS[m], label=m) for m in models_ordered]
ax2.legend(handles=legend_patches, fontsize=9)

plt.tight_layout(pad=2.5)
plt.savefig('efficiency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: efficiency_analysis.png')

## 6. RAFT Per-Task Breakdown (Qwen3-8B)

In [ ]:
RAFT_TASKS = [
    ('ADE Corpus v2',            0.14, 5000),
    ('Banking-77',               0.24, 5000),
    ('NeurIPS Impact Risks',     0.90,  150),
    ('One Stop English',         0.40,  516),
    ('Overruling',               0.96, 2350),
    ('Semiconductor Org Types',  0.02,  449),
    ('TAI Safety Research',      0.28, 1639),
    ('Terms of Service',         0.76, 5000),
    ('Tweet Eval Hate',          0.26, 2966),
    ('Twitter Complaints',       0.12, 3399),
]

raft_df = pd.DataFrame(RAFT_TASKS, columns=['task', 'accuracy', 'n_test'])
raft_df = raft_df.sort_values('accuracy', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: horizontal bar chart ────────────────────────────────────────────
ax = axes[0]
cmap = cm.get_cmap('RdYlGn')
bar_colors = [cmap(v) for v in raft_df['accuracy']]
bars = ax.barh(raft_df['task'], raft_df['accuracy'] * 100,
               color=bar_colors, edgecolor='white', linewidth=0.8)
mean_acc = raft_df['accuracy'].mean() * 100
ax.axvline(mean_acc, color='navy', linestyle='--', linewidth=1.5, label=f'Mean = {mean_acc:.1f}%')
for bar, v in zip(bars, raft_df['accuracy'] * 100):
    ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{v:.0f}%', va='center', ha='left', fontsize=9, fontweight='bold')
ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.set_title('RAFT Per-Task Accuracy\n(Qwen3-8B, 0-shot)', fontsize=11, fontweight='bold')
ax.set_xlim(0, 105)
ax.legend(fontsize=10)

# ── Right: bubble chart (accuracy vs. dataset size) ───────────────────────
ax2 = axes[1]
scatter = ax2.scatter(
    raft_df['n_test'], raft_df['accuracy'] * 100,
    s=raft_df['n_test'] / 20,
    c=raft_df['accuracy'], cmap='RdYlGn',
    alpha=0.85, edgecolors='gray', linewidths=0.7, zorder=5
)
for _, row in raft_df.iterrows():
    ax2.annotate(
        row['task'], (row['n_test'], row['accuracy'] * 100),
        xytext=(8, 4), textcoords='offset points', fontsize=8
    )
plt.colorbar(scatter, ax=ax2, label='Accuracy')
ax2.axhline(mean_acc, color='navy', linestyle='--', linewidth=1.5, alpha=0.7,
            label=f'Mean = {mean_acc:.1f}%')
ax2.set_xlabel('Dataset Size (# test examples)', fontsize=11)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.set_title('RAFT: Accuracy vs. Dataset Size\n(bubble size ∝ dataset size)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout(pad=2.5)
plt.savefig('raft_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: raft_breakdown.png')

## 7. Radar Chart — Model Capability Profile

In [ ]:
# We include tasks where ≥2 models have results
radar_tasks = [
    ('Financial PhraseBank', [0.906, 0.900, 0.963]),  # Llama8B, Qwen4B, Qwen8B
    ('GSM8K (Math)',         [0.7445, None, 0.8764]),
    ('BoolQ',                [0.8456, 0.8544, 0.8679]),
    ('RTE',                  [0.7834, 0.8664, 0.8592]),
    ('RAFT (mean)',           [None,   None,   0.408]),
]

labels = [t[0] for t in radar_tasks]
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

model_vals = {
    'Llama-3 8B': [0.906, 0.7445, 0.8456, 0.7834, None],
    'Qwen3 4B':   [0.900, None,   0.8544, 0.8664, None],
    'Qwen3 8B':   [0.963, 0.8764, 0.8679, 0.8592, 0.408],
}

for model, vals in model_vals.items():
    # Replace None with NaN and impute with task minimum for display
    filled = []
    for i, v in enumerate(vals):
        if v is None:
            # Use task minimum across known models so gaps don't distort shape
            known = [model_vals[m][i] for m in model_vals if model_vals[m][i] is not None]
            filled.append(min(known) if known else 0.5)
        else:
            filled.append(v)
    filled += filled[:1]  # close

    ax.plot(angles, filled, 'o-', linewidth=2, label=model, color=MODEL_COLORS[model])
    ax.fill(angles, filled, alpha=0.12, color=MODEL_COLORS[model])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], fontsize=8)
ax.set_title('Model Capability Radar Chart\n(0-shot Evaluation)', fontsize=13, fontweight='bold', pad=22)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.12), fontsize=10)

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: radar_chart.png')

## 8. Model Size vs. Accuracy (Pareto Efficiency)

In [ ]:
# Average accuracy per model across all tasks they participated in
mean_acc_by_model = (
    df[df['task'] != 'raft']  # exclude RAFT since only 1 model
    .groupby('model')['accuracy'].mean()
)
print('Mean accuracy (excluding RAFT) per model:')
for m, v in mean_acc_by_model.items():
    print(f'  {m}: {v*100:.2f}%')

mean_tps_by_model = (
    df.dropna(subset=['gen_tokens_per_sec'])
    .groupby('model')['gen_tokens_per_sec'].mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: Model size vs. mean accuracy ────────────────────────────────────
ax = axes[0]
for model in ['Llama-3 8B', 'Qwen3 4B', 'Qwen3 8B']:
    if model in mean_acc_by_model:
        ax.scatter(
            MODEL_SIZES[model], mean_acc_by_model[model] * 100,
            s=300, color=MODEL_COLORS[model], zorder=5,
            edgecolors='white', linewidths=1.5
        )
        ax.annotate(
            model,
            (MODEL_SIZES[model], mean_acc_by_model[model] * 100),
            xytext=(0, 12), textcoords='offset points',
            ha='center', fontsize=10, fontweight='bold', color=MODEL_COLORS[model]
        )

ax.set_xlabel('Model Size (Billions of Parameters)', fontsize=11)
ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
ax.set_title('Model Size vs. Mean Accuracy\n(0-shot, excl. RAFT)', fontsize=11, fontweight='bold')
ax.set_xlim(2, 10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))

# ── Right: Mean throughput vs. mean accuracy (Pareto plot) ────────────────
ax2 = axes[1]
for model in ['Llama-3 8B', 'Qwen3 4B', 'Qwen3 8B']:
    if model in mean_acc_by_model and model in mean_tps_by_model:
        ax2.scatter(
            mean_tps_by_model[model], mean_acc_by_model[model] * 100,
            s=300, color=MODEL_COLORS[model], zorder=5,
            edgecolors='white', linewidths=1.5
        )
        ax2.annotate(
            model,
            (mean_tps_by_model[model], mean_acc_by_model[model] * 100),
            xytext=(0, 12), textcoords='offset points',
            ha='center', fontsize=10, fontweight='bold', color=MODEL_COLORS[model]
        )

# Shade the Pareto-optimal region (upper-right = better)
ax2.set_xlabel('Mean Throughput (tokens/sec)', fontsize=11)
ax2.set_ylabel('Mean Accuracy (%)', fontsize=11)
ax2.set_title('Accuracy vs. Throughput (Pareto Plot)\n(upper-right corner = ideal)', fontsize=11, fontweight='bold')
ax2.set_xscale('log')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))

# Annotate ideal direction
ax2.annotate('← Slower    Faster →', xy=(0.5, 0.02), xycoords='axes fraction',
             ha='center', fontsize=9, color='gray')
ax2.annotate('Better ↑', xy=(0.02, 0.92), xycoords='axes fraction',
             ha='left', fontsize=9, color='gray', rotation=90)

plt.tight_layout(pad=2.5)
plt.savefig('pareto_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pareto_analysis.png')

## 9. Speedup Factor: Qwen3-8B vs. Llama-3-8B

In [ ]:
# Tasks where both models have throughput numbers
shared_tasks = ['financial_phrase_bank', 'superglue_boolq', 'superglue_rte']

speedup_data = []
for task in shared_tasks:
    llama_tps = df[(df['task'] == task) & (df['model'] == 'Llama-3 8B')]['gen_tokens_per_sec'].values
    qwen8_tps = df[(df['task'] == task) & (df['model'] == 'Qwen3 8B')]['gen_tokens_per_sec'].values
    qwen4_tps = df[(df['task'] == task) & (df['model'] == 'Qwen3 4B')]['gen_tokens_per_sec'].values

    if len(llama_tps) and len(qwen8_tps):
        speedup_data.append({
            'task': TASK_LABELS[task],
            'Qwen3 8B vs Llama-3 8B': qwen8_tps[0] / llama_tps[0],
            'Qwen3 4B vs Llama-3 8B': qwen4_tps[0] / llama_tps[0] if len(qwen4_tps) else np.nan,
        })

speedup_df = pd.DataFrame(speedup_data).set_index('task')

# Also add GSM8K: Qwen8B vs Llama8B
gsm_llama = df[(df['task'] == 'gsm8k') & (df['model'] == 'Llama-3 8B')]['gen_tokens_per_sec'].values
gsm_qwen8 = df[(df['task'] == 'gsm8k') & (df['model'] == 'Qwen3 8B')]['gen_tokens_per_sec'].values
if len(gsm_llama) and len(gsm_qwen8):
    speedup_df.loc['GSM8K (Math)', 'Qwen3 8B vs Llama-3 8B'] = gsm_qwen8[0] / gsm_llama[0]
    speedup_df.loc['GSM8K (Math)', 'Qwen3 4B vs Llama-3 8B'] = np.nan

print('=== Throughput Speedup vs. Llama-3 8B ===')
print(speedup_df.round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x2 = np.arange(len(speedup_df))
w = 0.35
b1 = ax.bar(x2 - w/2, speedup_df['Qwen3 8B vs Llama-3 8B'], w,
            label='Qwen3 8B vs Llama-3 8B', color=MODEL_COLORS['Qwen3 8B'], alpha=0.9)
b2 = ax.bar(x2 + w/2, speedup_df['Qwen3 4B vs Llama-3 8B'], w,
            label='Qwen3 4B vs Llama-3 8B', color=MODEL_COLORS['Qwen3 4B'], alpha=0.9)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, label='Baseline (1×)')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if not np.isnan(h) and h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                f'{h:.1f}×', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x2)
ax.set_xticklabels(speedup_df.index, fontsize=10)
ax.set_ylabel('Throughput Speedup Factor (×)', fontsize=11)
ax.set_title('vLLM Throughput Speedup vs. Llama-3 8B Baseline', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('speedup_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: speedup_analysis.png')

## 10. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax_acc   = fig.add_subplot(gs[0, :2])
ax_radar = fig.add_subplot(gs[0, 2], polar=True)
ax_eff   = fig.add_subplot(gs[1, :2])
ax_raft  = fig.add_subplot(gs[1, 2])

# ── (A) Accuracy bars ─────────────────────────────────────────────────────
for i, model in enumerate(models_ordered):
    vals = []
    for tl in tasks_ordered:
        row = df_main[(df_main['task_label'] == tl) & (df_main['model'] == model)]
        vals.append(row['accuracy'].values[0] * 100 if len(row) else np.nan)
    ax_acc.bar(x + offsets[i], vals, width,
               label=model, color=MODEL_COLORS[model], edgecolor='white', alpha=0.9)
ax_acc.set_xticks(x); ax_acc.set_xticklabels(tasks_ordered, fontsize=9)
ax_acc.set_ylabel('Accuracy (%)'); ax_acc.set_ylim(55, 105)
ax_acc.set_title('(A) Accuracy by Benchmark', fontweight='bold', fontsize=11)
ax_acc.legend(title='Model', fontsize=8, title_fontsize=8)
ax_acc.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

# ── (B) Radar ─────────────────────────────────────────────────────────────
for model, vals in model_vals.items():
    filled = []
    for i, v in enumerate(vals):
        if v is None:
            known = [model_vals[m][i] for m in model_vals if model_vals[m][i] is not None]
            filled.append(min(known) if known else 0.5)
        else:
            filled.append(v)
    filled += filled[:1]
    ax_radar.plot(angles, filled, 'o-', linewidth=1.8, label=model, color=MODEL_COLORS[model])
    ax_radar.fill(angles, filled, alpha=0.10, color=MODEL_COLORS[model])
ax_radar.set_xticks(angles[:-1]); ax_radar.set_xticklabels(labels, fontsize=7)
ax_radar.set_ylim(0, 1.05); ax_radar.set_yticks([0.4, 0.7, 1.0])
ax_radar.set_yticklabels(['40%', '70%', '100%'], fontsize=7)
ax_radar.set_title('(B) Capability Radar', fontweight='bold', fontsize=11, pad=14)

# ── (C) Efficiency bars ───────────────────────────────────────────────────
eff_data = df_eff.copy()
eff_data['acc_per_min'] = eff_data['accuracy'] / eff_data['total_gen_time_min']
eff_data = eff_data.sort_values('acc_per_min', ascending=True)
clrs = [MODEL_COLORS[m] for m in eff_data['model']]
ylabels = eff_data['task_label'] + ' (' + eff_data['model'] + ')'
ax_eff.barh(ylabels, eff_data['acc_per_min'], color=clrs, edgecolor='white')
ax_eff.set_xlabel('Accuracy / Minute'); ax_eff.set_title('(C) Inference Efficiency', fontweight='bold', fontsize=11)
legend_p = [mpatches.Patch(color=MODEL_COLORS[m], label=m) for m in models_ordered]
ax_eff.legend(handles=legend_p, fontsize=8)
ax_eff.tick_params(axis='y', labelsize=8)

# ── (D) RAFT breakdown ────────────────────────────────────────────────────
raft_sorted = raft_df.sort_values('accuracy')
raft_clrs = [cm.get_cmap('RdYlGn')(v) for v in raft_sorted['accuracy']]
ax_raft.barh(raft_sorted['task'], raft_sorted['accuracy'] * 100,
             color=raft_clrs, edgecolor='white')
ax_raft.axvline(raft_df['accuracy'].mean() * 100, color='navy',
                linestyle='--', linewidth=1.2, label=f"Mean {raft_df['accuracy'].mean()*100:.1f}%")
ax_raft.set_xlabel('Accuracy (%)'); ax_raft.set_title('(D) RAFT Task Breakdown\n(Qwen3-8B)', fontweight='bold', fontsize=11)
ax_raft.legend(fontsize=8); ax_raft.tick_params(axis='y', labelsize=7.5)

fig.suptitle('CS224N — 0-Shot Evaluation Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dashboard.png')

## 11. Key Findings

In [ ]:
print('=' * 65)
print('  CS224N 0-Shot Evaluation — Key Findings')
print('=' * 65)

print('\n📊 ACCURACY HIGHLIGHTS')
for model in ['Llama-3 8B', 'Qwen3 4B', 'Qwen3 8B']:
    sub = df[df['model'] == model]
    mean = sub['accuracy'].mean()
    best_task = sub.loc[sub['accuracy'].idxmax(), 'task_label']
    best_val  = sub['accuracy'].max()
    print(f'  {model:15s}  mean={mean*100:.1f}%  best={best_val*100:.1f}% ({best_task})')

print('\n⚡ THROUGHPUT HIGHLIGHTS')
df_t = df.dropna(subset=['gen_tokens_per_sec'])
fastest_row = df_t.loc[df_t['gen_tokens_per_sec'].idxmax()]
slowest_row = df_t.loc[df_t['gen_tokens_per_sec'].idxmin()]
print(f'  Fastest run: {fastest_row["model"]} on {fastest_row["task_label"]} '
      f'→ {fastest_row["gen_tokens_per_sec"]:.0f} tok/s')
print(f'  Slowest run: {slowest_row["model"]} on {slowest_row["task_label"]} '
      f'→ {slowest_row["gen_tokens_per_sec"]:.0f} tok/s')
if len(gsm_llama) and len(gsm_qwen8):
    speedup = gsm_qwen8[0] / gsm_llama[0]
    print(f'  GSM8K vLLM speedup (Qwen3-8B vs Llama-3-8B): {speedup:.1f}×')

print('\n🎯 MODEL RECOMMENDATIONS')
print('  • Best accuracy overall:    Qwen3-8B')
print('  • Best efficiency (acc/min): Qwen3-8B (vLLM batching)')
print('  • Best small-model tradeoff: Qwen3-4B (close accuracy, faster on simple tasks)')
print('  • Llama-3-8B falls behind on both accuracy AND throughput without vLLM')

print('\n⚠️  RAFT ANALYSIS')
print(f'  Mean RAFT accuracy (Qwen3-8B, 0-shot): {raft_df["accuracy"].mean()*100:.1f}%')
print(f'  Highest task: Overruling ({0.96*100:.0f}%)')
print(f'  Lowest task:  Semiconductor Org Types ({0.02*100:.0f}%)')
print(f'  Std dev across tasks: {raft_df["accuracy"].std()*100:.1f}% — high variance!')
print('  → 0-shot RAFT performance is highly task-dependent;')
print('    few-shot prompting may be critical for low-performing tasks.')
print('=' * 65)